# CISLR — Final Deployment Pipeline

## Stage 9.3 — Canonical OpenVINO Prototype Retrieval Engine

### Purpose

This notebook begins the final deployment and system-integration phase of the
CISLR sign-language recognition project.

The preceding experimental stages are treated as frozen scientific evidence.
This notebook does not retrain, modify, or tune the previously evaluated models.

Stage 9.3 integrates the validated 960-dimensional OpenVINO MobileNetV3-Large
embedding extractor with the canonical Stage 4.5 CISLR prototype gallery and
verifies end-to-end prototype retrieval on real CISLR videos.

---

## Frozen upstream evidence

The following stages are considered completed and are not modified here:

- Stage 4 — lightweight spatial-model comparison and MobileNetV3-Large selection
- Stage 4.5 — official CISLR prototype-to-test retrieval evaluation
- Stage 4.7–4.9 — OpenVINO conversion, benchmarking, and deployment evaluation
- Stage 5 — full-frame versus Hand-ROI controlled ablation
- Stage 6 — temporal-representation experiments
- Stage 7 — retrieval and post-processing experiments
- Stage 8 — corrected scientific/development analysis
- Stage 9.0 — final artifact compatibility verification
- Stage 9.1 — 960-D OpenVINO embedding-model export
- Stage 9.2 — real-CISLR-video OpenVINO embedding validation

---

## Stage 9.3 objective

The deployment retrieval path is:

Real CISLR video  
→ prescribed 8-frame sampling  
→ image preprocessing  
→ OpenVINO MobileNetV3-Large  
→ 960-D frame embeddings  
→ mean temporal pooling  
→ L2 normalization  
→ cosine similarity against the frozen prototype gallery  
→ Top-k CISLR gloss retrieval

Stage 9.3 must reproduce the historical Stage 4.5 retrieval behaviour before
the recognition engine is connected to a live webcam.

---

## Scientific / Methodological Disclaimer

This stage is a deployment-integration and reproducibility experiment.

No model training, fine-tuning, test-label tuning, prototype adaptation, or
test-set optimization is performed in Stage 9.3.

The Stage 4.5 prototype embeddings, metadata, and historical predictions are
treated as frozen reference artifacts.

The Stage 9.1 OpenVINO model is used only as an inference-time embedding
extractor. Stage 9.2 established that this model reproduces the historical
Stage 4.5 MobileNetV3-Large 960-D embedding representation on sampled real
CISLR videos.

The canonical Stage 4.5 prototype population is used in this stage rather than
silently substituting prototype banks created by later controlled-subset or
real-time demonstration experiments.

Any difference between newly generated OpenVINO retrieval results and the
historical Stage 4.5 results will be reported rather than manually corrected.

This stage evaluates isolated-sign prototype retrieval. It must not be
interpreted as continuous sign-language translation or sentence-level
translation.

---

## Success criteria

Stage 9.3 is considered successful when:

1. The frozen Stage 4.5 prototype gallery loads correctly.
2. Prototype embeddings and prototype metadata are exactly aligned.
3. The OpenVINO extractor produces 960-D embeddings.
4. Real CISLR test videos can be processed end-to-end.
5. Top-1 and Top-5 predictions can be produced using cosine similarity.
6. New OpenVINO retrieval predictions agree closely with the corresponding
   historical Stage 4.5 retrieval results.
7. All validation artifacts are saved without modifying upstream artifacts.

Only after this validation passes will the retrieval engine proceed to
offline deployment testing and live webcam integration.

In [1]:
# ==================================================================================================
# CISLR — STAGE 9.3
# CANONICAL OPENVINO PROTOTYPE RETRIEVAL ENGINE
# ==================================================================================================
#
# PURPOSE
# -------
# 1. Load the validated Stage 9.1 OpenVINO 960-D embedding extractor.
# 2. Load the canonical Stage 4.5 MobileNetV3-Large prototype gallery.
# 3. Verify prototype/metadata alignment.
# 4. Select real Stage 4.5 test videos.
# 5. Recreate their 8-frame video embeddings using OpenVINO.
# 6. Perform cosine-similarity retrieval against ALL canonical prototypes.
# 7. Compare new Top-1 / Top-5 retrieval with historical Stage 4.5 predictions.
# 8. Save reproducibility evidence.
#
# IMPORTANT
# ---------
# - NO TRAINING
# - NO FINE-TUNING
# - NO TEST-LABEL TUNING
# - NO PROTOTYPE MODIFICATION
# - NO HAND-ROI
# - NO TTA / WHITENING / CSLS
# - NO WEBCAM YET
#
# ==================================================================================================

from pathlib import Path
import ast
import json
import time

import cv2
import numpy as np
import pandas as pd

from openvino import Core


# ==================================================================================================
# 0. PATHS
# ==================================================================================================

ROOT = Path("/home/dipshikha/Music/cao/CISLR_RESEARCH")

DEPLOYMENT_ROOT = ROOT / "CISLR_FINAL_DEPLOYMENT"

STAGE9_ROOT = ROOT / "audit" / "stage9_final_openvino"

STAGE9_3_DIR = STAGE9_ROOT / "retrieval_engine"
REPORT_DIR = STAGE9_3_DIR / "reports"
PREDICTION_DIR = STAGE9_3_DIR / "predictions"

DEPLOYMENT_ROOT.mkdir(parents=True, exist_ok=True)
STAGE9_3_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
PREDICTION_DIR.mkdir(parents=True, exist_ok=True)


# Validated Stage 9.1 OpenVINO embedding model
OV_XML = (
    STAGE9_ROOT
    / "embedding_fp32"
    / "mobilenet_v3_large_embedding_960_fp32_static.xml"
)


# Canonical Stage 4.5 artifacts
STAGE45_EMBED_DIR = (
    ROOT
    / "audit"
    / "stage4_5_official_evaluation"
    / "embeddings"
)

STAGE45_PRED_DIR = (
    ROOT
    / "audit"
    / "stage4_5_official_evaluation"
    / "predictions"
)

PROTOTYPE_EMBED_PATH = (
    STAGE45_EMBED_DIR
    / "mobilenetv3_large_prototype_embeddings.npy"
)

PROTOTYPE_META_PATH = (
    STAGE45_EMBED_DIR
    / "mobilenetv3_large_prototype_metadata.csv"
)

TEST_META_PATH = (
    STAGE45_EMBED_DIR
    / "mobilenetv3_large_test_metadata.csv"
)

HISTORICAL_PRED_PATH = (
    STAGE45_PRED_DIR
    / "mobilenetv3_large_official_predictions.csv"
)


print("=" * 100)
print("CISLR — STAGE 9.3")
print("CANONICAL OPENVINO PROTOTYPE RETRIEVAL ENGINE")
print("=" * 100)

required_files = {
    "OpenVINO model": OV_XML,
    "Prototype embeddings": PROTOTYPE_EMBED_PATH,
    "Prototype metadata": PROTOTYPE_META_PATH,
    "Test metadata": TEST_META_PATH,
    "Historical predictions": HISTORICAL_PRED_PATH,
}

print("\nARTIFACT CHECK")
print("-" * 100)

for name, path in required_files.items():
    print(f"{name:25s}: {path}")
    print(f"{'':25s}  Exists = {path.exists()}")

missing = [str(p) for p in required_files.values() if not p.exists()]

if missing:
    raise FileNotFoundError(
        "Required Stage 9.3 artifacts are missing:\n" + "\n".join(missing)
    )

print("\n[OK] All required artifacts exist.")


# ==================================================================================================
# 1. LOAD CANONICAL STAGE 4.5 GALLERY
# ==================================================================================================

prototype_bank = np.load(PROTOTYPE_EMBED_PATH).astype(np.float32)
prototype_meta = pd.read_csv(PROTOTYPE_META_PATH)

test_meta = pd.read_csv(TEST_META_PATH)
historical_pred = pd.read_csv(HISTORICAL_PRED_PATH)


print("\n" + "=" * 100)
print("CANONICAL STAGE 4.5 GALLERY")
print("=" * 100)

print("Prototype bank shape :", prototype_bank.shape)
print("Prototype metadata   :", prototype_meta.shape)
print("Test metadata        :", test_meta.shape)
print("Historical predictions:", historical_pred.shape)


if prototype_bank.ndim != 2:
    raise ValueError(
        f"Prototype bank must be 2-D. Got {prototype_bank.shape}"
    )

if prototype_bank.shape[1] != 960:
    raise ValueError(
        f"Expected 960-D prototype embeddings. Got {prototype_bank.shape[1]}"
    )

if len(prototype_bank) != len(prototype_meta):
    raise ValueError(
        "Prototype embedding/metadata count mismatch: "
        f"{len(prototype_bank)} vs {len(prototype_meta)}"
    )


# Normalize defensively for cosine retrieval.
prototype_norms_before = np.linalg.norm(
    prototype_bank,
    axis=1,
    keepdims=True
)

prototype_bank = (
    prototype_bank
    / np.clip(prototype_norms_before, 1e-12, None)
).astype(np.float32)


prototype_norms_after = np.linalg.norm(
    prototype_bank,
    axis=1
)

print("\nPrototype norm after normalization:")
print(" min :", float(prototype_norms_after.min()))
print(" mean:", float(prototype_norms_after.mean()))
print(" max :", float(prototype_norms_after.max()))

print("\n[OK] Canonical Stage 4.5 gallery ready.")


# ==================================================================================================
# 2. DETECT IMPORTANT COLUMN NAMES
# ==================================================================================================

print("\nPrototype metadata columns:")
print(list(prototype_meta.columns))

print("\nTest metadata columns:")
print(list(test_meta.columns))

print("\nHistorical prediction columns:")
print(list(historical_pred.columns))


required_proto_columns = {"uid", "gloss"}

if not required_proto_columns.issubset(prototype_meta.columns):
    raise ValueError(
        f"Prototype metadata must contain {required_proto_columns}"
    )

required_test_columns = {
    "uid",
    "gloss",
    "video_path",
    "sample_indices",
}

if not required_test_columns.issubset(test_meta.columns):
    raise ValueError(
        f"Test metadata must contain {required_test_columns}"
    )


# ==================================================================================================
# 3. LOAD OPENVINO MODEL
# ==================================================================================================

core = Core()

ov_model = core.read_model(str(OV_XML))
compiled_model = core.compile_model(ov_model, "CPU")

input_layer = compiled_model.input(0)
output_layer = compiled_model.output(0)

print("\n" + "=" * 100)
print("OPENVINO MODEL")
print("=" * 100)

print("Input shape :", tuple(input_layer.shape))
print("Output shape:", tuple(output_layer.shape))


if tuple(input_layer.shape) != (1, 3, 224, 224):
    raise ValueError(
        f"Unexpected OpenVINO input shape: {tuple(input_layer.shape)}"
    )

if tuple(output_layer.shape) != (1, 960):
    raise ValueError(
        f"Unexpected OpenVINO output shape: {tuple(output_layer.shape)}"
    )

print("\n[OK] OpenVINO model ready.")


# ==================================================================================================
# 4. PREPROCESSING
# ==================================================================================================

IMAGENET_MEAN = np.array(
    [0.485, 0.456, 0.406],
    dtype=np.float32
).reshape(1, 1, 3)

IMAGENET_STD = np.array(
    [0.229, 0.224, 0.225],
    dtype=np.float32
).reshape(1, 1, 3)


def preprocess_frame(frame_bgr):
    """
    Reproduce the preprocessing validated during Stage 9.2.
    """

    frame_rgb = cv2.cvtColor(
        frame_bgr,
        cv2.COLOR_BGR2RGB
    )

    frame_rgb = cv2.resize(
        frame_rgb,
        (224, 224),
        interpolation=cv2.INTER_LINEAR
    )

    x = frame_rgb.astype(np.float32) / 255.0

    x = (x - IMAGENET_MEAN) / IMAGENET_STD

    x = np.transpose(x, (2, 0, 1))

    x = np.expand_dims(x, axis=0)

    return np.ascontiguousarray(
        x,
        dtype=np.float32
    )


# ==================================================================================================
# 5. FRAME INDEX PARSER
# ==================================================================================================

def parse_sample_indices(value):

    if isinstance(value, (list, tuple, np.ndarray)):
        return [int(v) for v in value]

    if pd.isna(value):
        raise ValueError("sample_indices is missing.")

    parsed = ast.literal_eval(str(value))

    return [int(v) for v in parsed]


# ==================================================================================================
# 6. EXACT FRAME READING
# ==================================================================================================

def read_exact_frames(video_path, frame_indices):

    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        raise RuntimeError(
            f"Could not open video: {video_path}"
        )

    frames = []

    try:

        for idx in frame_indices:

            cap.set(
                cv2.CAP_PROP_POS_FRAMES,
                int(idx)
            )

            ok, frame = cap.read()

            if not ok or frame is None:
                raise RuntimeError(
                    f"Could not decode frame {idx} "
                    f"from {video_path}"
                )

            frames.append(frame)

    finally:
        cap.release()

    if len(frames) != len(frame_indices):
        raise RuntimeError(
            f"Expected {len(frame_indices)} frames, "
            f"decoded {len(frames)}"
        )

    return frames


# ==================================================================================================
# 7. FRAME EMBEDDING
# ==================================================================================================

def extract_frame_embedding(frame_bgr):

    x = preprocess_frame(frame_bgr)

    result = compiled_model([x])

    embedding = np.asarray(
        result[output_layer]
    ).reshape(-1)

    if embedding.shape != (960,):
        raise ValueError(
            f"Unexpected frame embedding shape: "
            f"{embedding.shape}"
        )

    return embedding.astype(np.float32)


# ==================================================================================================
# 8. VIDEO EMBEDDING
# ==================================================================================================

def extract_video_embedding(
    video_path,
    frame_indices
):

    frames = read_exact_frames(
        video_path,
        frame_indices
    )

    frame_embeddings = []

    inference_times_ms = []

    for frame in frames:

        start = time.perf_counter()

        embedding = extract_frame_embedding(frame)

        elapsed_ms = (
            time.perf_counter() - start
        ) * 1000.0

        frame_embeddings.append(embedding)
        inference_times_ms.append(elapsed_ms)

    frame_embeddings = np.stack(
        frame_embeddings,
        axis=0
    )

    # Frozen temporal aggregation:
    # simple mean pooling.
    video_embedding = frame_embeddings.mean(
        axis=0
    )

    norm = np.linalg.norm(video_embedding)

    if norm <= 1e-12:
        raise RuntimeError(
            "Video embedding has zero norm."
        )

    video_embedding = (
        video_embedding / norm
    ).astype(np.float32)

    return (
        video_embedding,
        frame_embeddings,
        inference_times_ms,
    )


# ==================================================================================================
# 9. COSINE RETRIEVAL
# ==================================================================================================

def retrieve_topk(
    query_embedding,
    k=5
):

    # Both query and gallery are normalized.
    # Dot product therefore equals cosine similarity.

    similarities = (
        prototype_bank @ query_embedding
    )

    k = min(k, len(similarities))

    top_indices = np.argpartition(
        -similarities,
        k - 1
    )[:k]

    top_indices = top_indices[
        np.argsort(
            -similarities[top_indices]
        )
    ]

    results = []

    for rank, idx in enumerate(
        top_indices,
        start=1
    ):

        row = prototype_meta.iloc[int(idx)]

        results.append(
            {
                "rank": rank,
                "prototype_index": int(idx),
                "prototype_uid": str(row["uid"]),
                "gloss": str(row["gloss"]),
                "similarity": float(
                    similarities[idx]
                ),
            }
        )

    return results


# ==================================================================================================
# 10. HISTORICAL PREDICTION LOOKUP
# ==================================================================================================

def get_historical_row(uid):

    if "uid" not in historical_pred.columns:
        return None

    match = historical_pred[
        historical_pred["uid"].astype(str)
        == str(uid)
    ]

    if len(match) == 0:
        return None

    return match.iloc[0]


# ==================================================================================================
# 11. SELECT VALIDATION VIDEOS
# ==================================================================================================

if "status" in test_meta.columns:

    valid_test = test_meta[
        test_meta["status"].astype(str).str.upper()
        == "OK"
    ].copy()

else:
    valid_test = test_meta.copy()


# Use a deterministic small validation sample first.
# Stage 9.4 can later evaluate a larger population.

N_VALIDATION = 10

validation_rows = valid_test.head(
    N_VALIDATION
).copy()


print("\n" + "=" * 100)
print("STAGE 9.3 VALIDATION SAMPLE")
print("=" * 100)

print(
    validation_rows[
        ["uid", "gloss", "video_path"]
    ].to_string(index=False)
)


# ==================================================================================================
# 12. RUN RETRIEVAL
# ==================================================================================================

summary_rows = []
topk_rows = []


for sample_number, (_, row) in enumerate(
    validation_rows.iterrows(),
    start=1
):

    uid = str(row["uid"])
    true_gloss = str(row["gloss"])
    video_path = Path(str(row["video_path"]))

    frame_indices = parse_sample_indices(
        row["sample_indices"]
    )

    print("\n" + "=" * 100)
    print(
        f"[{sample_number}/{len(validation_rows)}] "
        f"{uid} | TRUE = {true_gloss}"
    )
    print("=" * 100)

    print("Video :", video_path)
    print("Frames:", frame_indices)

    if not video_path.exists():

        print("[ERROR] Video does not exist.")

        summary_rows.append(
            {
                "uid": uid,
                "true_gloss": true_gloss,
                "status": "VIDEO_MISSING",
            }
        )

        continue


    try:

        (
            query_embedding,
            frame_embeddings,
            inference_times_ms,
        ) = extract_video_embedding(
            video_path,
            frame_indices
        )


        retrieval = retrieve_topk(
            query_embedding,
            k=5
        )


        predicted_top1 = retrieval[0]["gloss"]

        top5_glosses = [
            item["gloss"]
            for item in retrieval
        ]


        top1_correct = (
            predicted_top1 == true_gloss
        )

        top5_correct = (
            true_gloss in top5_glosses
        )


        print("\nOPENVINO RETRIEVAL")

        for item in retrieval:

            marker = (
                " <-- TRUE"
                if item["gloss"] == true_gloss
                else ""
            )

            print(
                f"{item['rank']:>2}. "
                f"{item['gloss']:<35} "
                f"{item['similarity']:.6f}"
                f"{marker}"
            )


        historical_row = get_historical_row(uid)

        historical_top1 = None
        historical_top1_match = None

        if historical_row is not None:

            possible_top1_columns = [
                "top1_gloss",
                "predicted_gloss",
                "prediction",
                "pred_gloss",
            ]

            for col in possible_top1_columns:

                if col in historical_pred.columns:

                    historical_top1 = str(
                        historical_row[col]
                    )

                    break


            if historical_top1 is not None:

                historical_top1_match = (
                    historical_top1
                    == predicted_top1
                )


        print("\nRESULT")
        print("True gloss          :", true_gloss)
        print("OpenVINO Top-1      :", predicted_top1)
        print("Top-1 correct       :", top1_correct)
        print("Top-5 contains true :", top5_correct)

        if historical_top1 is not None:
            print(
                "Historical Top-1    :",
                historical_top1
            )
            print(
                "Historical agreement:",
                historical_top1_match
            )


        mean_frame_latency = float(
            np.mean(inference_times_ms)
        )

        p95_frame_latency = float(
            np.percentile(
                inference_times_ms,
                95
            )
        )


        summary_rows.append(
            {
                "uid": uid,
                "true_gloss": true_gloss,
                "status": "OK",

                "openvino_top1":
                    predicted_top1,

                "top1_correct":
                    bool(top1_correct),

                "top5_correct":
                    bool(top5_correct),

                "historical_top1":
                    historical_top1,

                "historical_top1_agreement":
                    historical_top1_match,

                "top1_similarity":
                    retrieval[0]["similarity"],

                "mean_frame_inference_ms":
                    mean_frame_latency,

                "p95_frame_inference_ms":
                    p95_frame_latency,
            }
        )


        for item in retrieval:

            topk_rows.append(
                {
                    "query_uid": uid,
                    "true_gloss": true_gloss,

                    "rank":
                        item["rank"],

                    "retrieved_uid":
                        item["prototype_uid"],

                    "retrieved_gloss":
                        item["gloss"],

                    "similarity":
                        item["similarity"],

                    "is_true_gloss":
                        item["gloss"]
                        == true_gloss,
                }
            )


    except Exception as exc:

        print(
            "[ERROR]",
            type(exc).__name__,
            str(exc)
        )

        summary_rows.append(
            {
                "uid": uid,
                "true_gloss": true_gloss,
                "status":
                    f"ERROR: {type(exc).__name__}: {exc}",
            }
        )


# ==================================================================================================
# 13. BUILD RESULTS
# ==================================================================================================

summary_df = pd.DataFrame(
    summary_rows
)

topk_df = pd.DataFrame(
    topk_rows
)


print("\n" + "=" * 100)
print("STAGE 9.3 RESULTS")
print("=" * 100)

print(summary_df.to_string(index=False))


# ==================================================================================================
# 14. METRICS
# ==================================================================================================

ok_df = summary_df[
    summary_df["status"] == "OK"
].copy()


if len(ok_df) == 0:
    raise RuntimeError(
        "No Stage 9.3 videos were successfully evaluated."
    )


top1_accuracy = float(
    ok_df["top1_correct"].mean()
)

top5_accuracy = float(
    ok_df["top5_correct"].mean()
)


agreement_values = (
    ok_df["historical_top1_agreement"]
    .dropna()
)

if len(agreement_values) > 0:

    historical_agreement = float(
        agreement_values.astype(bool).mean()
    )

else:

    historical_agreement = None


mean_latency = float(
    ok_df["mean_frame_inference_ms"].mean()
)

mean_p95_latency = float(
    ok_df["p95_frame_inference_ms"].mean()
)


print("\n" + "-" * 100)
print("SUMMARY")
print("-" * 100)

print(
    f"Videos requested          : "
    f"{len(validation_rows)}"
)

print(
    f"Videos successfully tested: "
    f"{len(ok_df)}"
)

print(
    f"Sample Top-1 accuracy     : "
    f"{top1_accuracy:.4f}"
)

print(
    f"Sample Top-5 accuracy     : "
    f"{top5_accuracy:.4f}"
)

print(
    f"Historical Top-1 agreement: "
    f"{historical_agreement}"
)

print(
    f"Mean frame inference      : "
    f"{mean_latency:.3f} ms"
)

print(
    f"Mean per-video p95 frame  : "
    f"{mean_p95_latency:.3f} ms"
)


# ==================================================================================================
# 15. SAVE RESULTS
# ==================================================================================================

SUMMARY_CSV = (
    PREDICTION_DIR
    / "stage9_3_retrieval_summary.csv"
)

TOPK_CSV = (
    PREDICTION_DIR
    / "stage9_3_top5_predictions.csv"
)

REPORT_JSON = (
    REPORT_DIR
    / "stage9_3_retrieval_report.json"
)


summary_df.to_csv(
    SUMMARY_CSV,
    index=False
)

topk_df.to_csv(
    TOPK_CSV,
    index=False
)


report = {

    "stage": "9.3",

    "name":
        "Canonical OpenVINO Prototype Retrieval Engine",

    "status":
        "COMPLETE",

    "openvino_model":
        str(OV_XML),

    "prototype_embedding_source":
        str(PROTOTYPE_EMBED_PATH),

    "prototype_metadata_source":
        str(PROTOTYPE_META_PATH),

    "historical_prediction_source":
        str(HISTORICAL_PRED_PATH),

    "prototype_count":
        int(len(prototype_bank)),

    "embedding_dimension":
        int(prototype_bank.shape[1]),

    "videos_requested":
        int(len(validation_rows)),

    "videos_successfully_evaluated":
        int(len(ok_df)),

    "sample_top1_accuracy":
        top1_accuracy,

    "sample_top5_accuracy":
        top5_accuracy,

    "historical_top1_agreement":
        historical_agreement,

    "mean_frame_inference_ms":
        mean_latency,

    "mean_per_video_p95_frame_inference_ms":
        mean_p95_latency,

    "methodology": {

        "frame_sampling":
            "historical Stage 4.5 sample_indices",

        "frame_count":
            8,

        "spatial_model":
            "Stage 9.1 OpenVINO MobileNetV3-Large 960-D extractor",

        "temporal_aggregation":
            "mean pooling",

        "normalization":
            "L2",

        "retrieval":
            "cosine similarity",

        "gallery":
            "canonical Stage 4.5 prototype gallery",

        "hand_roi":
            False,

        "test_label_tuning":
            False,

        "tta":
            False,

        "whitening":
            False,

        "csls":
            False,
    },
}


with open(
    REPORT_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        report,
        f,
        indent=2
    )


print("\nSaved:")
print(SUMMARY_CSV)
print(TOPK_CSV)
print(REPORT_JSON)


# ==================================================================================================
# 16. FINAL STAGE DECISION
# ==================================================================================================

print("\n" + "=" * 100)

if len(ok_df) == len(validation_rows):

    print("STAGE 9.3 — EXECUTION COMPLETE")

    print(
        "\nThe canonical OpenVINO retrieval engine "
        "successfully processed all requested real CISLR videos."
    )

    if historical_agreement is not None:

        print(
            "\nHistorical Top-1 agreement:",
            f"{historical_agreement:.4f}"
        )

    print(
        "\nIMPORTANT:"
        "\nDo not interpret the small validation sample's accuracy "
        "as the official CISLR accuracy."
        "\nThe official research metrics remain those from the "
        "frozen Stage 4.5 evaluation."
    )

else:

    print("STAGE 9.3 — COMPLETE WITH ERRORS")

    print(
        "\nAt least one requested video failed."
        "\nInspect the saved results before proceeding."
    )

print("=" * 100)

CISLR — STAGE 9.3
CANONICAL OPENVINO PROTOTYPE RETRIEVAL ENGINE

ARTIFACT CHECK
----------------------------------------------------------------------------------------------------
OpenVINO model           : /home/dipshikha/Music/cao/CISLR_RESEARCH/audit/stage9_final_openvino/embedding_fp32/mobilenet_v3_large_embedding_960_fp32_static.xml
                           Exists = True
Prototype embeddings     : /home/dipshikha/Music/cao/CISLR_RESEARCH/audit/stage4_5_official_evaluation/embeddings/mobilenetv3_large_prototype_embeddings.npy
                           Exists = True
Prototype metadata       : /home/dipshikha/Music/cao/CISLR_RESEARCH/audit/stage4_5_official_evaluation/embeddings/mobilenetv3_large_prototype_metadata.csv
                           Exists = True
Test metadata            : /home/dipshikha/Music/cao/CISLR_RESEARCH/audit/stage4_5_official_evaluation/embeddings/mobilenetv3_large_test_metadata.csv
                           Exists = True
Historical predictions   : /home/